# Blinded Exposure-Retrieval Benchmark

This notebook tests whether holdings embeddings function as an economic search engine. For each geopolitical channel, it retrieves ten nonseed firms using the all-S12 asset embedding and pairs them with ten firms matched on industry, size, beta, value, momentum, profitability, and investment. Candidate origin is then hidden before relationship validation.

**No event returns are loaded anywhere in this notebook.** The output is a frozen, randomized validation queue—not a portfolio or event-performance result.

In [ ]:
from pathlib import Path
import hashlib,json,sys
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if ROOT.name!='geopolitical_exposure': ROOT=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'retrieval_benchmark';PUBLIC=OUTPUT/'public';RESTRICTED=OUTPUT/'restricted_truth'
PUBLIC.mkdir(parents=True,exist_ok=True);RESTRICTED.mkdir(parents=True,exist_ok=True)
from src.retrieval_benchmark import freeze_blinded_queue,build_evidence_packets,queue_hash
K_PER_CHANNEL=10;RANDOM_SEED=20220224
print('Embedding and control candidates per channel:',K_PER_CHANNEL)

## 1. Freeze the design

The primary estimand is the difference in independently verified relationship precision between embedding retrieval and matched controls. Candidate selection, matching, evidence retrieval, prompts, and adjudication must be frozen before unblinding. Ukraine is still a development event, but return outcomes remain quarantined from this experiment.

In [ ]:
design={'event':'Russia invasion of Ukraine','information_cutoff':'2022-02-23','representation':'all_s12_rs_l_min_10d','k_per_channel':K_PER_CHANNEL,'random_seed':RANDOM_SEED,'primary_metric':'precision_at_k_embedding_minus_matched_control','outcomes_loaded':False,'relationship_types':['peer_or_competitor','supplier','customer','substitute','complement','geographic_operations','commodity_producer','commodity_consumer','financial_exposure','common_investor_clientele_only','unsupported','ambiguous']}
design['sha256']=hashlib.sha256(json.dumps(design,sort_keys=True).encode()).hexdigest()
(PUBLIC/'design.json').write_text(json.dumps(design,indent=2));display(pd.DataFrame([design]))

## 2. Select embedding candidates and matched controls

Direct text seeds are excluded from both groups. Every control is selected without replacement. A four-point distance penalty strongly favors exact SIC2 matches while standardized continuous characteristics determine the nearest match within the available pool.

In [ ]:
public_queue,hidden_mapping,matches=freeze_blinded_queue(ROOT,K_PER_CHANNEL,RANDOM_SEED)
public_queue.to_parquet(PUBLIC/'blinded_candidate_queue.parquet',index=False)
hidden_mapping.to_parquet(RESTRICTED/'candidate_source_mapping.parquet',index=False)
matches.to_parquet(RESTRICTED/'embedding_control_pairs.parquet',index=False)
display(matches.groupby('pair_channel').agg(pairs=('pair_rank','size'),exact_sic2_rate=('same_sic2','mean'),median_match_distance=('match_distance','median'),maximum_match_distance=('match_distance','max')).reset_index())
display(public_queue)

## 3. Verify blindness and balance

The public queue contains no retrieval-source field, similarity score, propagated score, event return, or portfolio assignment. The restricted map must not be opened during relationship adjudication.

In [ ]:
forbidden=['hidden_source','similarity','propagated','car_','return','portfolio']
leaks=[column for column in public_queue.columns if any(term in column.lower() for term in forbidden)]
assert not leaks,f'Blinding failure: {leaks}'
assert len(public_queue)==2*5*K_PER_CHANNEL
assert public_queue.blind_id.is_unique
balance=pd.DataFrame({'metric':['blinded_rows','channels','firms','duplicate_firm_channel_rows','hidden_embedding_rows','hidden_control_rows'],'value':[len(public_queue),public_queue.channel.nunique(),public_queue.permco.nunique(),public_queue.duplicated(['permco','channel']).sum(),hidden_mapping.hidden_source.eq('embedding').sum(),hidden_mapping.hidden_source.eq('matched_control').sum()]})
display(balance)

## 4. Build identical pre-event evidence packets

For each blinded candidate, the latest available pre-event SEC filing is divided into overlapping chunks. TF-IDF retrieves eight passages using the frozen channel vocabulary, relationship terms, and the same three reference seeds assigned to both members of an embedding-control pair. Retrieval is mechanical and never uses outcomes.

In [ ]:
packets=build_evidence_packets(ROOT,public_queue,passages=8)
packet_path=PUBLIC/'blinded_evidence_packets.jsonl'
with packet_path.open('w') as handle:
    for packet in packets: handle.write(json.dumps(packet)+'\n')
packet_audit=pd.DataFrame([{'blind_id':p['blind_id'],'channel':p['channel'],'document_available':p['document_available'],'passages':len(p.get('passages',[])),'retrieved_characters':sum(len(x['text']) for x in p.get('passages',[])),'latest_available_date':p.get('available_date')} for p in packets])
packet_audit.to_parquet(PUBLIC/'evidence_packet_audit.parquet',index=False)
display(packet_audit.groupby('channel').agg(candidates=('blind_id','size'),documents=('document_available','sum'),median_passages=('passages','median'),median_characters=('retrieved_characters','median')).reset_index())

## 5. Create the blinded adjudication template

Validation must determine whether pre-event evidence supports a real relationship to the reference seeds and channel. `unsupported` is a valid and important answer. Reviewers must not infer a relationship from company names, industries, embedding proximity, or outside post-event knowledge.

In [ ]:
review=public_queue[['blind_id','permco','firm_name','ticker','channel','reference_seed_names']].copy()
for column,default in [('relationship_type','pending'),('relationship_direction','pending'),('confidence',np.nan),('evidence_quote',''),('document_id',''),('evidence_start',pd.NA),('evidence_end',pd.NA),('adjudication_decision','pending'),('review_notes','')]: review[column]=default
review.to_csv(PUBLIC/'blinded_relationship_review_template.csv',index=False)
display(review.head(20))

## 6. Freeze the queue contract

The hash below binds the randomized candidate identities to the experiment. The next stage may use the same structured OpenAI extraction infrastructure, but responses must require exact pre-event evidence and remain blind to candidate origin and returns. A human must adjudicate accepted relationships before the restricted source mapping is opened.

In [ ]:
contract={'design_sha256':design['sha256'],'public_queue_sha256':queue_hash(public_queue),'packet_sha256':hashlib.sha256(packet_path.read_bytes()).hexdigest(),'queued_candidate_channel_rows':len(public_queue),'candidate_origin_blinded':True,'event_outcomes_loaded':False,'unblind_condition':'all relationship decisions finalized and evidence quotes verified'}
contract['sha256']=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
(PUBLIC/'retrieval_contract.json').write_text(json.dumps(contract,indent=2));display(pd.DataFrame([contract]))

## 7. Stop before unblinding

Do not open `restricted_truth/candidate_source_mapping.parquet` yet. The next notebook stage will run or import blinded relationship classifications, verify every quotation against its source document, finalize unsupported and ambiguous decisions, and only then reveal candidate origin to calculate Precision@K and matched-pair lift. No event returns are needed for that evaluation.